<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У нас є користувачі, які:

* встановлюють застосунок (`app_sessions`),
* переглядають товари (`product_views_log`),
* авторизуються (`devices_users_map`),
* здійснюють покупки (`orders_log`).

# Робота з датами у SQL
**Основні функції для роботи з датою при аналізі даних**

## `DATE_TRUNC`  
**обрізає дату або час до заданої одиниці виміру**, обнуляючи менш значущі компоненти.

### Загальний синтаксис:

```sql
DATE_TRUNC('одиниця_часу', timestamp)
```

---

### Для чого використовується:

1. **Групування у часі**
   Наприклад, по тижнях, місяцях, днях тощо.
2. **Округлення до початку години/дня і т.д.**
3. **Порівняння дат з потрібною точністю**
   (наприклад, всі події за конкретний місяць).

---

### Підтримувані одиниці:

* `millennium`, `century`, `decade`, `year`, `quarter`, `month`
* `week`, `day`, `hour`, `minute`, `second`

In [1]:
DB_USER = "prog_academy_da_157g_user"
DB_PASSWORD = "XTE8IHOEtQZ5GasTkaOjcGMKYTha6LeO"
DB_HOST = "dpg-d8t957gg4nts73da5bg0-a.frankfurt-postgres.render.com"
DB_PORT = "5432"
DB_NAME = "prog_academy_da_157g"


url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"


from sqlalchemy import create_engine

engine = create_engine(url)

import pandas as pd


In [6]:
query = """
SELECT DATE_TRUNC('month', first_seen)
FROM app_sessions;
"""

df = pd.read_sql(query, engine)
df

,date_trunc
0,2024-04-01 00:00:00+00:00
1,2024-01-01 00:00:00+00:00
2,2024-01-01 00:00:00+00:00
3,2024-02-01 00:00:00+00:00
4,2024-04-01 00:00:00+00:00
...,...
495,2024-06-01 00:00:00+00:00
496,2024-02-01 00:00:00+00:00
497,2024-01-01 00:00:00+00:00
498,2024-02-01 00:00:00+00:00


## `EXTRACT`
**витягує окрему частину дати або часу**, наприклад: рік, місяць, день, година і т.д.

### Синтаксис:

```sql
EXTRACT(одиниця_часу FROM дата_або_час)
```

---

### Використовувані одиниці:

* `YEAR`
* `MONTH`
* `DAY`
* `HOUR`
* `MINUTE`
* `SECOND`
* `DOW` (day of week: 0 = Sunday, 6 = Saturday)
* `DOY` (day of year)
* `WEEK`
* `QUARTER`

---

### Для чого застосовується:

1. **Фільтрація по частині дати**
2. **Створення часових групувань**

---

### Різниця з `DATE_TRUNC`:

|                | `EXTRACT`                     | `DATE_TRUNC`                                 |
| -------------- | ----------------------------- | -------------------------------------------- |
| Що робить      | Повертає одне число           | Обрізає менш значущі частини дати            |
| Тип результату | `numeric`                     | `timestamp`                                  |
| Приклад        | `EXTRACT(MONTH FROM d)` → `7` | `DATE_TRUNC('month', d)` → `'2025-07-01...'` |

In [9]:
query = """
SELECT EXTRACT(doy FROM first_seen)
FROM app_sessions;
"""

df = pd.read_sql(query, engine)
df

,extract
0,116.0
1,15.0
2,18.0
3,54.0
4,115.0
...,...
495,172.0
496,39.0
497,10.0
498,42.0


## `CURRENT_DATE`, `CURRENT_TIMESTAMP` и `NOW()`
**поточна дата та/або час**

### Особенности

| Функція             | Що повертає                     | Тип даних   | Приклад                      |
| ------------------- | ------------------------------- | ----------- | ---------------------------- |
| `CURRENT_DATE`      | Лише поточну дату               | `DATE`      | `2025-07-14`                 |
| `CURRENT_TIMESTAMP` | Поточну дату **та** час         | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |
| `NOW()`             | Те саме, що `CURRENT_TIMESTAMP` | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |


In [11]:
query = """
SELECT CURRENT_DATE, CURRENT_TIMESTAMP, NOW()
"""

df = pd.read_sql(query, engine)
df

,current_date,current_timestamp,now
0,2026-07-07,2026-07-07 17:19:37.505858+00:00,2026-07-07 17:19:37.505858+00:00


## `INTERVAL`  
**додавання/віднімання часових проміжків** (дні, години, місяці, секунди і т.д.)

### 📌 Синтаксис:

```sql
SELECT CURRENT_DATE + INTERVAL '7 days';
```
---

### Одиниці:

* `second`, `minute`, `hour`
* `day`, `week`, `month`, `year`
* * комбінації, наприклад:

```sql
INTERVAL '2 days 3 hours 15 minutes'
```
---

### Особливості:

* `INTERVAL` сам по собі — не дата, це **часовий відрізок**
* Використовується з `DATE` і з `TIMESTAMP`
* В MySQL запис трохи інший:

```sql
-- MySQL стиль
SELECT NOW() + INTERVAL 1 DAY;
```
---

### В Pandas аналог:

```python
import pandas as pd

pd.Timestamp.now() + pd.Timedelta(days=3, hours=2)
```

In [16]:
query = """
SELECT *
FROM app_sessions
WHERE first_seen >= NOW() - INTERVAL '830 days';
"""

df = pd.read_sql(query, engine)
df

,session_id,device_code,first_seen,os_type,acquisition_channel,cpi_uah
0,S00000,DVC0000,2024-04-25,Android,Facebook,30.53
1,S00004,DVC0004,2024-04-24,Android,Google Ads,29.64
2,S00005,DVC0005,2024-06-02,Android,Facebook,10.67
3,S00007,DVC0007,2024-05-25,iOS,Organic,15.28
4,S00009,DVC0009,2024-05-05,Android,Referral,21.54
...,...,...,...,...,...,...
256,S00491,DVC0491,2024-05-28,iOS,Referral,27.90
257,S00492,DVC0492,2024-04-21,Android,Organic,31.41
258,S00494,DVC0494,2024-04-18,iOS,Facebook,16.96
259,S00495,DVC0495,2024-06-20,iOS,Organic,19.58


## Коли застосовується робота з датами та часом

| Ціль                                      | Коли застосовуємо                                  | Що потрібно                      |
| ----------------------------------------- | -------------------------------------------------- | -------------------------------- |
| **1. Графіки по днях / тижнях / місяцях** | Динаміка показників у часі                         | Групування по даті               |
| **2. Сезонність**                         | Піки/спади у поведінці                             | Групування по дню тижня, місяцю  |
| **3. Повторюваність покупок**             | Як часто повертаються користувачі                  | Обчислення інтервалів між датами |
| **4. Retention**                          | Чи повертаються користувачі після першої взаємодії | Когорти за першою подією         |

### Завдання 1. Встановлення по місяцях
Порахуйте кількість встановлень застосунку (`app_sessions`) по кожному місяцю. Використайте `DATE_TRUNC('month', first_seen)`, відсортуйте результат за місяцем.
Очікувані колонки: `month`, `installs_count`.

### Завдання 2. Розподіл замовлень по днях тижня
За допомогою `EXTRACT(DOW FROM order_time)` визначте, у які дні тижня користувачі роблять найбільше замовлень. Виведіть номер дня тижня, кількість замовлень і сумарну виручку `total_uah`, відсортовану за кількістю замовлень.

### Завдання 3. Час до першої покупки
Для кожного користувача, який здійснив хоча б одну покупку, порахуйте кількість днів між датою встановлення застосунку (`first_seen` пристрою, з якого прийшов користувач) та датою його першого замовлення (`MIN(order_time)`). Виведіть `user_uuid`, `first_seen`, `first_order_date`, `days_to_first_order`.
Підказка: з'єднайте `app_sessions` → `devices_users_map` → `orders_log`, згрупуйте по користувачу.

### Завдання 4. Перегляди без покупки протягом тижня
Знайдіть перегляди товарів (`product_views_log`), після яких користувач **не** зробив жодного замовлення протягом 7 днів від дати перегляду (`view_date + INTERVAL '7 days'`). Виведіть `device_code`, `view_date` для таких переглядів.
Підказка: `LEFT JOIN` на `orders_log` через `devices_users_map` з умовою по діапазону дат, залишити рядки без збігу (`IS NULL`).

### Завдання 5. Воронка "встановлення → перегляд → покупка" по каналах
Для кожного `acquisition_channel` порахуйте:
- кількість встановлень (`installs`),
- кількість унікальних пристроїв, які зробили хоча б один перегляд (`viewed_devices`),
- кількість унікальних користувачів, які зробили хоча б одну покупку (`buyers`).

Виведіть також частку `viewed_devices / installs` та `buyers / installs` (конверсію воронки).

### Завдання 6. CPI, виручка та ROMI по каналах
Для кожного `acquisition_channel` розрахуйте:
- сумарні витрати на залучення: `SUM(cpi_uah)`,
- сумарну виручку від користувачів цього каналу: `SUM(total_uah)` по всіх їхніх замовленнях,
- ROMI (Return on Marketing Investment) за формулою `(revenue - spend) / spend * 100`.

Відсортуйте канали за ROMI за спаданням.

### Завдання 7. ARPU та ARPPU по каналах
Для кожного `acquisition_channel` порахуйте:
- **ARPU** (Average Revenue Per User) — виручка / кількість усіх встановлень каналу,
- **ARPPU** (Average Revenue Per Paying User) — виручка / кількість користувачів, які хоча б раз заплатили

### Завдання 8. Середній чек і частота покупок по платформах
Використовуючи `os_type` з `app_sessions`, для кожної платформи порахуйте:
- середній чек (`AVG(total_uah)`) по всіх замовленнях,
- середню кількість замовлень на одного платного користувача

### Завдання 9. Частка користувачів із повторною покупкою та рейтинг каналів
Порахуйте частку користувачів, які зробили **більше однієї** покупки, серед усіх користувачів (`repeat_purchase_rate`) — загалом по всій базі.